This file covers: advanced spark examples


In [ ]:
from pyspark.sql import SparkSession, functions as func
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

spark = SparkSession.builder.appName('SuperHeroes').getOrCreate()

schema = StructType([
    StructField('id', IntegerType()),
    StructField('name', StringType())
])

names = spark.read.schema(schema).option('sep', ' ').csv('./MarvelNames.txt')

lines = spark.read.text('MarvelGraph.txt')

connections = lines.withColumn('id', func.split(func.col('value'), ' ')[0]) \
.withColumn('connections', func.size(func.split(func.col('value'), ' ')) - 1) \
.groupBy('id').agg(func.sum('connections').alias('connections'))

mostPopular = connections.sort(func.col('connections').desc()).first()

mostPopularName = names.filter(func.col('id') == mostPopular[0]).select('name').first()

print(mostPopularName[0] + ' is the most popular superhero with ' + str(mostPopular[1]) + ' co-appearences.')

spark.stop()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/23 15:47:44 WARN Utils: Your hostname, Rev-PF3CHFLT, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/23 15:47:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/23 15:47:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


BERSERKER II is the least popular superhero with 1 co-appearences.


In [1]:
# NOW DO LEAST POPULAR

from pyspark.sql import SparkSession, functions as func
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

spark = SparkSession.builder.appName('SuperHeroes').getOrCreate()

schema = StructType([
    StructField('id', IntegerType()),
    StructField('name', StringType())
])

names = spark.read.schema(schema).option('sep', ' ').csv('./MarvelNames.txt')

lines = spark.read.text('MarvelGraph.txt')

connections = lines.withColumn('id', func.split(func.col('value'), ' ')[0]) \
.withColumn('connections', func.size(func.split(func.col('value'), ' ')) - 1) \
.groupBy('id').agg(func.sum('connections').alias('connections'))

minConnectionCount = connections.agg(func.min('connections')).first()[0]

minConnections = connections.filter(func.col('connections') == minConnectionCount)

minConnectionsWithNames = minConnections.join(names, 'id')

print('The following characters only have ' + str(minConnectionCount) + ' connection(s):')
minConnectionsWithNames.select('name').show()

spark.stop()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/23 15:54:31 WARN Utils: Your hostname, Rev-PF3CHFLT, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/23 15:54:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/23 15:54:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


The following characters only have 1 connection(s):
+--------------------+
|                name|
+--------------------+
|        BERSERKER II|
|              BLARE/|
|MARVEL BOY II/MARTIN|
|MARVEL BOY/MARTIN BU|
|      GIURESCU, RADU|
|       CLUMSY FOULUP|
|              FENRIS|
|              RANDAK|
|           SHARKSKIN|
|     CALLAHAN, DANNY|
|         DEATHCHARGE|
|                RUNE|
|         SEA LEOPARD|
|         RED WOLF II|
|              ZANTOR|
|JOHNSON, LYNDON BAIN|
|          LUNATIK II|
|                KULL|
|GERVASE, LADY ALYSSA|
+--------------------+

